# Phase 3 — Model Training & Evaluation

**Goal:** Train a baseline MLP emotion classifier on MFCC features and evaluate with accuracy, F1-score, confusion matrix.

## Steps
1. Train model + 5-fold cross-validation
2. Confusion matrix
3. Per-class F1 bar chart
4. Learning curve
5. Error analysis (most confused pairs)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
EXPORTS_DIR = Path('../data/monitoring_exports')
MODELS_DIR  = Path('../models')

## 1. Train Model

In [ ]:
from src.model import train

results = train(
    config_path='../configs/config.yaml',
    features_dir='../data/features',
    models_dir='../models',
    exports_dir='../data/monitoring_exports',
    feature_type='mfcc',
    run_cv=True,
)

# Or from terminal:
# python src/model.py
# python src/model.py --feature_type wav2vec2

## 2. Confusion Matrix

In [ ]:
cm_df = pd.read_csv(EXPORTS_DIR / 'confusion_matrix.csv', index_col=0)

# Normalize by row (true label)
cm_norm = cm_df.div(cm_df.sum(axis=1), axis=0)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.5, vmin=0, vmax=1,
    xticklabels=cm_df.columns, yticklabels=cm_df.index
)
plt.title('Confusion Matrix (row-normalized)', fontsize=13)
plt.ylabel('True Emotion')
plt.xlabel('Predicted Emotion')
plt.tight_layout()
plt.savefig(EXPORTS_DIR / 'confusion_matrix_plot.png', dpi=120)
plt.show()

## 3. Per-Class F1 Score

In [ ]:
report_df = pd.read_csv(EXPORTS_DIR / 'classification_report.csv')

# Keep only emotion rows (exclude macro avg, weighted avg, accuracy)
emotions = ['neutral','calm','happy','sad','angry','fearful','disgust','surprised']
report_emotions = report_df[report_df['label'].isin(emotions)].copy()
report_emotions = report_emotions.sort_values('f1_score', ascending=True)

COLORS = {
    'neutral':'#95a5a6','calm':'#3498db','happy':'#f1c40f','sad':'#2980b9',
    'angry':'#e74c3c','fearful':'#9b59b6','disgust':'#27ae60','surprised':'#e67e22'
}
bar_colors = [COLORS[e] for e in report_emotions['label']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(report_emotions['label'], report_emotions['f1_score'],
               color=bar_colors, edgecolor='white')
ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=10)
ax.set_xlim(0, 1.1)
ax.axvline(report_emotions['f1_score'].mean(), color='red',
           linestyle='--', label=f"Mean F1: {report_emotions['f1_score'].mean():.3f}")
ax.legend()
ax.set_title('Per-Class F1 Score (Baseline Model)', fontsize=13)
ax.set_xlabel('F1 Score')
plt.tight_layout()
plt.savefig(EXPORTS_DIR / 'per_class_f1.png', dpi=120)
plt.show()

## 4. Learning Curve

In [ ]:
import joblib
import json
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import learning_curve

model  = joblib.load(MODELS_DIR / 'mlp_model.joblib')
scaler = joblib.load(MODELS_DIR / 'scaler.joblib')

X = np.load('../data/features/mfcc_features.npy')
y = np.load('../data/features/labels.npy')
meta = pd.read_csv('../data/features/metadata.csv')
train_mask = meta['split'] == 'train'
X_tr = scaler.transform(X[train_mask.values])
y_tr = y[train_mask.values]

train_sizes, train_scores, val_scores = learning_curve(
    model, X_tr, y_tr,
    cv=3, scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#e74c3c', label='Train F1')
plt.fill_between(train_sizes,
                 train_scores.mean(1) - train_scores.std(1),
                 train_scores.mean(1) + train_scores.std(1), alpha=0.1, color='#e74c3c')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#3498db', label='Val F1')
plt.fill_between(train_sizes,
                 val_scores.mean(1) - val_scores.std(1),
                 val_scores.mean(1) + val_scores.std(1), alpha=0.1, color='#3498db')
plt.title('Learning Curve (F1-macro)', fontsize=13)
plt.xlabel('Training Samples')
plt.ylabel('F1-macro')
plt.legend()
plt.tight_layout()
plt.savefig(EXPORTS_DIR / 'learning_curve.png', dpi=120)
plt.show()

## 5. Error Analysis — Most Confused Emotion Pairs

In [ ]:
preds_df = pd.read_csv(EXPORTS_DIR / 'predictions.csv')
errors   = preds_df[~preds_df['correct']].copy()

# Count confusion pairs
pairs = errors.groupby(['emotion', 'predicted']).size().reset_index(name='count')
pairs = pairs.sort_values('count', ascending=False).head(10)
pairs['pair'] = pairs['emotion'] + ' → ' + pairs['predicted']

plt.figure(figsize=(10, 5))
plt.barh(pairs['pair'], pairs['count'], color='#e74c3c', edgecolor='white')
plt.title('Top 10 Most Confused Emotion Pairs', fontsize=13)
plt.xlabel('Error Count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(EXPORTS_DIR / 'error_analysis.png', dpi=120)
plt.show()

print(f'\nTotal test errors: {len(errors)} / {len(preds_df)} ({len(errors)/len(preds_df):.1%})')

## 6. Baseline Summary

In [ ]:
metrics = pd.read_csv(EXPORTS_DIR / 'model_metrics.csv').iloc[-1]

print('=' * 40)
print('      BASELINE MODEL SUMMARY')
print('=' * 40)
print(f"  Feature type : {metrics['feature_type']}")
print(f"  Train Acc    : {metrics['train_acc']:.4f}")
print(f"  Test  Acc    : {metrics['test_acc']:.4f}")
print(f"  Train F1     : {metrics['train_f1']:.4f}")
print(f"  Test  F1     : {metrics['test_f1']:.4f}")
if pd.notna(metrics.get('cv_f1_mean')):
    print(f"  CV F1        : {metrics['cv_f1_mean']:.4f} ± {metrics['cv_f1_std']:.4f}")
print('=' * 40)
print('Baseline locked. Phase 4 will monitor drift from this point.')